## Data Load

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Daily_Water_Intake.csv')
df.head(5)

,Age,Gender,Weight (kg),Daily Water Intake (liters),Physical Activity Level,Weather,Hydration Level
0,56,Male,96,4.23,Moderate,Hot,Good
1,60,Male,105,3.95,High,Normal,Good
2,36,Male,68,2.39,Moderate,Cold,Good
3,19,Female,74,3.13,Moderate,Hot,Good
4,38,Male,77,2.11,Low,Normal,Poor


In [4]:
df.shape

(30000, 7)

In [5]:
df.columns

Index(['Age', 'Gender', 'Weight (kg)', 'Daily Water Intake (liters)',
       'Physical Activity Level', 'Weather', 'Hydration Level'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Age                          30000 non-null  int64  
 1   Gender                       30000 non-null  object 
 2   Weight (kg)                  30000 non-null  int64  
 3   Daily Water Intake (liters)  30000 non-null  float64
 4   Physical Activity Level      30000 non-null  object 
 5   Weather                      30000 non-null  object 
 6   Hydration Level              30000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 1.6+ MB


## Data Cleaning and encoding

In [7]:
# encode target variable (classification problem)
# good = 0 not dehydrated
# poor = 1 dehydrated unhealthy

In [9]:
df['Hydration_Label'] = df['Hydration Level'].map({ 'Good' : 0, 'Poor': 1})

In [10]:
df = df.drop(columns=['Hydration Level']) # not needed now

In [11]:
df.head(5)

,Age,Gender,Weight (kg),Daily Water Intake (liters),Physical Activity Level,Weather,Hydration_Label
0,56,Male,96,4.23,Moderate,Hot,0
1,60,Male,105,3.95,High,Normal,0
2,36,Male,68,2.39,Moderate,Cold,0
3,19,Female,74,3.13,Moderate,Hot,0
4,38,Male,77,2.11,Low,Normal,1


In [14]:
# Encode Categorical Features
# Gender, Physical Activity Level, Weather

In [16]:
# One-Hot Encoding

df_encoded = pd.get_dummies(
    df,
    columns=['Gender', 'Physical Activity Level', 'Weather'],
    drop_first=True
)

## drop_first = True. why ? Avoids multicollinearity, Cleaner models

In [18]:
# Separate Features & Targets

X = df_encoded.drop(columns=['Hydration_Label'])
y = df_encoded['Hydration_Label']

### Train-Test Split

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
X_train.shape, X_test.shape

((24000, 8), (6000, 8))

In [22]:
y.value_counts(normalize=True)

Hydration_Label
0    0.797167
1    0.202833
Name: proportion, dtype: float64

## Train Dehydration Detection Models (Classification)

### 1. Logistic Regression baseline model
#### Feature Scaling

In [30]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [31]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced', random_state=42)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

In [32]:
from sklearn.metrics import classification_report, confusion_matrix

print("Logistic Regression Results:\n")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Results:

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      4783
           1       0.93      1.00      0.96      1217

    accuracy                           0.98      6000
   macro avg       0.96      0.99      0.98      6000
weighted avg       0.99      0.98      0.98      6000

[[4690   93]
 [   0 1217]]


<b> <p>The Logistic Regression model achieved 98% accuracy and 100% recall for poor hydration, ensuring no dehydrated cases were missed. The model maintains a 0.93 precision for dehydration detection, indicating a small number of false alerts while prioritizing safety.

ZERO false negatives
- No dehydrated person is missed
- This is perfect for health applications </p>
</b>

### 2. Random Forest 

In [40]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [41]:
print("Random Forest Results:\n")
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))


Random Forest Results:

              precision    recall  f1-score   support

           0       0.99      0.93      0.96      4783
           1       0.79      0.98      0.87      1217

    accuracy                           0.94      6000
   macro avg       0.89      0.96      0.92      6000
weighted avg       0.95      0.94      0.94      6000

[[4461  322]
 [  26 1191]]


<b><p>Although Random Forest achieved 94% accuracy, it missed 26 dehydration cases and produced a higher number of false alerts. Logistic Regression outperformed Random Forest with 98% accuracy and 100% recall for dehydration detection, making it more suitable for this application.

- 1191 dehydrated cases detected
- 26 dehydrated cases missed (false negatives)
- 322 hydrated people falsely flagged as dehydrated
- Lower precision for class 1
</p></b>

#### Feature importance

In [35]:
import pandas as pd

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance.head(10)


Daily Water Intake (liters)         0.323546
Weight (kg)                         0.197883
Physical Activity Level_Low         0.195812
Weather_Hot                         0.173538
Age                                 0.041949
Physical Activity Level_Moderate    0.041103
Weather_Normal                      0.024858
Gender_Male                         0.001310
dtype: float64

<b>Daily water intake and environmental conditions were the strongest predictors of dehydration risk.</b>

## Water Intake Prediction (Regression)